In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
data = pd.read_csv(os.path.join('../data', 'merged_covid_and_opinion_data.csv'), parse_dates=['Day'], index_col='Day')
idx = data.index[data.notna().all(axis=1)]
new_cases = data.loc[idx, 'daily-new-confirmed-covid-19-cases-per-million-people'].rolling(window=7).mean()
new_deaths = data.loc[idx, 'daily-new-confirmed-covid-19-deaths-per-million-people'].rolling(window=7).mean()
data = data.drop(["daily-new-confirmed-covid-19-cases-per-million-people","daily-new-confirmed-covid-19-deaths-per-million-people"], axis = 1)
data = data.drop(["COVID-Like_Symptoms", "People_Wearing_Masks", "Tested_Positive", "Tested", "COVID-19_Vaccine_Acceptance__Vaccinated_Appointment_or_Accept", "COVID-Like_Symptoms_in_Community"], axis=1)

In [3]:
data["Public_Transit"] = 100 - data["Public_Transit"]
data["Restaurant_Indoors"] = 100 - data["Restaurant_Indoors"]
data["Shop_Indoors"] = 100 - data["Shop_Indoors"]
data.loc[idx] = data.loc[idx] / 100

In [6]:
from scipy.stats import bernoulli

labels = [0, 1, 2, 3, 4]
dist_data = pd.DataFrame(0, index=idx, columns=labels)
for index, row in data.loc[idx].iterrows():
    dist = bernoulli.rvs(row.Shop_Indoors, size=1000) + bernoulli.rvs(row.Worried_About_Catching_COVID, size=1000) + bernoulli.rvs(row.Public_Transit, size=1000) + bernoulli.rvs(row.Anxious, size=1000)
    res = np.unique(dist, return_counts=True)
    dist_data.loc[index, res[0]] = res[1]

In [10]:
dist_data.rename(columns={0:-2, 1:-1, 2:0, 3:1, 4:2}, inplace=True)
dist_data.to_csv("../data/mock_data.csv")